# Topic Modeling of Policy-Cited Literature

This notebook performs topic modeling on the cleaned publication
datasets produced by the data-preparation workflow.

The analysis includes text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, and temporal analysis.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile

# Project directories
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Output directory  :", OUTPUT_DIR)

Project directory : m:\projects_latex\review_paper_2
Data directory    : m:\projects_latex\review_paper_2\data
Output directory  : m:\projects_latex\review_paper_2\output


## 1. Load Cleaned Publication Data

The cleaned Overton and Scopus publication datasets produced by the
data-preparation notebook are loaded separately. The two sources are
retained as distinct datasets rather than merged.

In [4]:
OVERTON_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton = pd.read_excel(
    OVERTON_FILE
)

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Overton")
print("-------")
print(f"Documents : {len(overton):,}")
print(f"Columns   : {len(overton.columns):,}")

print("\nScopus")
print("------")
print(f"Documents : {len(scopus):,}")
print(f"Columns   : {len(scopus.columns):,}")

Overton
-------
Documents : 14,266
Columns   : 21

Scopus
------
Documents : 16,403
Columns   : 18


### 1.1 Define the Independent Topic-Modeling Corpora

The Overton and Scopus publication collections are analyzed as
independent corpora. Each corpus therefore receives its own text
preprocessing, document-term matrix, LDA model-selection procedure,
final topic model, and downstream topic analysis.

In [5]:
corpora = {
    "overton": overton.copy(),
    "scopus": scopus.copy(),
}

for corpus_name, corpus_df in corpora.items():

    print(f"{corpus_name.upper()}")
    print("-" * len(corpus_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print()

OVERTON
-------
Documents          : 14,266
Non-missing titles : 14,266
Non-missing abstracts: 14,264
Missing abstracts  : 2

SCOPUS
------
Documents          : 16,403
Non-missing titles : 16,403
Non-missing abstracts: 16,403
Missing abstracts  : 0



## 2. Prepare Corpora for Topic Modeling

Topic modeling is performed independently for the Overton and Scopus
datasets. Publications without abstracts are excluded because the LDA
models are estimated from abstract text.

In [6]:
lda_corpora = {}

for corpus_name, corpus_df in corpora.items():

    lda_df = (
        corpus_df[
            corpus_df["Abstract"].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Remove abstracts that are empty after whitespace stripping.
    lda_df["Abstract"] = (
        lda_df["Abstract"]
        .astype(str)
        .str.strip()
    )

    lda_df = (
        lda_df[
            lda_df["Abstract"] != ""
        ]
        .reset_index(drop=True)
    )

    lda_corpora[corpus_name] = lda_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))
    print(
        f"Input publications : "
        f"{len(corpus_df):,}"
    )
    print(
        f"LDA documents      : "
        f"{len(lda_df):,}"
    )
    print(
        f"Excluded           : "
        f"{len(corpus_df) - len(lda_df):,}"
    )
    print()
    

OVERTON
-------
Input publications : 14,266
LDA documents      : 14,264
Excluded           : 2

SCOPUS
------
Input publications : 16,403
LDA documents      : 16,403
Excluded           : 0



## 3. R Text-Processing Backend

The abstract corpora are preprocessed using R `tm` and `SnowballC`
through `Rscript`. This preserves the text-processing methodology used
for the LDA analysis while allowing the complete workflow to be
controlled from Python.

In [7]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"C:\Program Files\R\R-4.5.2\bin\Rscript.exe"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

Rscript:
C:\Program Files\R\R-4.5.2\bin\Rscript.exe

Required R packages:
tm=TRUE
SnowballC=TRUE
slam=TRUE
topicmodels=TRUE



### 3.1 Text-Preprocessing Configuration

The Overton and Scopus corpora are processed using an identical text
preprocessing configuration to support direct comparison between the
two independently estimated topic models.

Standard English stopwords are supplemented with terms appearing
explicitly in the literature-search query because these terms define
the corpus but provide limited information for distinguishing latent
topics.

In [ ]:
# Search-query terms removed from both corpora.
# change this based on your topic

CUSTOM_STOPWORDS = [
    # Search-query terms
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",

    # Publisher/copyright boilerplate
    "©",
]

# Initial DTM configuration.
MIN_TERM_LENGTH = 3
MIN_DOC_FREQ = 3

print("Query-specific stopwords:")
for word in QUERY_STOPWORDS:
    print(f"  - {word}")

print("\nDTM configuration")
print("-----------------")
print("Minimum term length     :", MIN_TERM_LENGTH)
print("Minimum document freq.  :", MIN_DOC_FREQ)

Query-specific stopwords:
  - power
  - flow
  - machine
  - learning
  - optimization
  - optimisation

DTM configuration
-----------------
Minimum term length     : 3
Minimum document freq.  : 3


### 3.2 Preprocess Abstracts with R `tm`

The same R `tm` and `SnowballC` preprocessing pipeline is applied
independently to the Overton and Scopus abstracts. Processing includes
lowercasing, punctuation and number removal, whitespace normalization,
English and query-specific stopword removal, and English Snowball
stemming.

The resulting corpora are used to inspect vocabulary characteristics
before the final document-term matrices are constructed.

In [9]:
R_PREPROCESS_DIR = OUTPUT_DIR / "r_preprocessing"

R_PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR / "preprocess_corpus.R"
)

r_preprocess_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Load abstracts
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

abstracts <- data$Abstract

# ------------------------------------------------------------
# Shared stopwords
# ------------------------------------------------------------

query_stops <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation"
)

all_stops <- unique(
    c(
        tm::stopwords("english"),
        query_stops
    )
)

# ------------------------------------------------------------
# tm preprocessing
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(abstracts)
)

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    all_stops
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

processed <- vapply(
    corpus,
    as.character,
    character(1)
)

result <- data.frame(
    document_id = seq_along(processed),
    processed_text = processed,
    stringsAsFactors = FALSE
)

write.csv(
    result,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created R preprocessing script:",
    R_PREPROCESS_SCRIPT
)

Created R preprocessing script: m:\projects_latex\review_paper_2\output\r_preprocessing\preprocess_corpus.R


In [10]:
processed_corpora = {}

for corpus_name, corpus_df in lda_corpora.items():

    print(
        f"Processing {corpus_name.upper()} ..."
    )

    input_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    # Only the abstract is needed by the R preprocessing backend.
    corpus_df[
        ["Abstract"]
    ].to_csv(
        input_file,
        index=False,
        encoding="utf-8",
    )

    run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_PREPROCESS_SCRIPT),
            str(input_file),
            str(output_file),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    processed_df = pd.read_csv(
        output_file,
        keep_default_na=False,
    )

    processed_corpora[
        corpus_name
    ] = processed_df

    print(
        f"Documents processed : "
        f"{len(processed_df):,}"
    )

    print(
        f"Empty documents     : "
        f"{(processed_df['processed_text'].str.strip() == '').sum():,}"
    )

    print()

Processing OVERTON ...
Documents processed : 14,264
Empty documents     : 1

Processing SCOPUS ...
Documents processed : 16,403
Empty documents     : 0



In [11]:
for corpus_name, processed_df in processed_corpora.items():

    token_counts = (
        processed_df[
            "processed_text"
        ]
        .str.split()
        .str.len()
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{len(processed_df):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(token_counts.sum()):,}"
    )

    print(
        f"Minimum tokens  : "
        f"{int(token_counts.min()):,}"
    )

    print(
        f"Median tokens   : "
        f"{token_counts.median():.0f}"
    )

    print(
        f"Mean tokens     : "
        f"{token_counts.mean():.1f}"
    )

    print(
        f"Maximum tokens  : "
        f"{int(token_counts.max()):,}"
    )

    print()

OVERTON
-------
Documents       : 14,264
Total tokens    : 1,584,965
Minimum tokens  : 0
Median tokens   : 108
Mean tokens     : 111.1
Maximum tokens  : 566

SCOPUS
------
Documents       : 16,403
Total tokens    : 2,075,071
Minimum tokens  : 2
Median tokens   : 124
Mean tokens     : 126.5
Maximum tokens  : 453



### 3.3 Inspect Frequent Terms

The most frequent terms remaining after preprocessing are inspected
before constructing the final document-term matrices. This diagnostic
is used to identify high-frequency generic terms that provide little
thematic discrimination and may therefore warrant inclusion in the
shared custom stopword list.

In [12]:
from collections import Counter

term_frequency_tables = {}

for corpus_name, processed_df in processed_corpora.items():

    term_frequency = Counter(
        token
        for text in processed_df["processed_text"]
        for token in text.split()
    )

    top_terms = pd.DataFrame(
        term_frequency.most_common(50),
        columns=[
            "term",
            "frequency",
        ],
    )

    term_frequency_tables[
        corpus_name
    ] = top_terms

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        top_terms.head(30)
    )


OVERTON
-------


,term,frequency
0,use,15161
1,system,14469
2,model,13986
3,©,12923
4,energi,9977
5,can,7645
6,data,7553
7,result,7509
8,studi,7246
9,paper,6678



SCOPUS
------


,term,frequency
0,system,32853
1,energi,22024
2,propos,21979
3,algorithm,21225
4,model,21219
5,method,18906
6,use,17585
7,©,16137
8,network,14453
9,distribut,13961
